# Cairn - LoRA Fine-Tuning Pipeline

Fine-tunes Llama-3.2-1B for all 6 Cairn domain agents using LoRA.

**Setup**: Runtime > Change runtime type > T4 GPU

**Time**: ~3 hours for all 6 domains

In [ ]:
# Step 0: Check GPU
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Step 1: Install dependencies
!pip install -q torch transformers safetensors tqdm huggingface_hub

In [ ]:
# Step 2: Clone repo
!git clone https://github.com/mavericaks/Cairn.git /content/cairn
%cd /content/cairn/ml-pipeline
!ls -la

In [ ]:
# Step 3: HuggingFace login
import os
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'  # <-- PASTE YOUR TOKEN

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print('Logged in.')

In [ ]:
# Step 4: Generate training datasets
!python generate_dataset.py --output_dir data

import os
print('\nDatasets:')
for f in sorted(os.listdir('data')):
    if f.endswith('.jsonl'):
        count = sum(1 for _ in open(f'data/{f}'))
        print(f'  {f}: {count} examples')

In [ ]:
# Step 5: Verify model loads correctly
from model import LlamaForCausalLM, LlamaConfig
from lora import inject_lora
import torch

config = LlamaConfig()
m = LlamaForCausalLM(config)
m = inject_lora(m, rank=16, alpha=32, target_modules=['q_proj', 'v_proj'])

ids = torch.randint(0, 100, (2, 8))
mask = torch.ones(2, 8, dtype=torch.long)
labels = ids.clone()
labels[:, :4] = -100
logits, loss = m(ids, attention_mask=mask, labels=labels)
print(f'Model test: logits={logits.shape}, loss type={type(loss)}')
print('Model structure OK!')
del m, logits, loss
import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Step 6: Train ALL domains
# ~25-40 min per domain on T4

DOMAINS = ['analytical', 'execution', 'discovery', 'generative', 'conversational', 'system']

import time

for domain in DOMAINS:
    start = time.time()
    print(f'\n{"="*60}')
    print(f'  TRAINING: {domain}')
    print(f'{"="*60}\n')

    !python train.py \
        --domain {domain} \
        --dataset data/{domain}.jsonl \
        --epochs 5 \
        --batch_size 4 \
        --grad_accum 4 \
        --lr 2e-4 \
        --max_length 512 \
        --lora_rank 16 \
        --lora_alpha 32

    elapsed = (time.time() - start) / 60
    print(f'  {domain} done in {elapsed:.1f} min')

    # Free memory between domains
    import gc; gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Step 7: Verify adapters were saved
import os
print('Saved adapters:')
for f in sorted(os.listdir('adapters')):
    size = os.path.getsize(f'adapters/{f}') / 1e6
    print(f'  {f}: {size:.1f} MB')

In [ ]:
# Step 8: Export merged models
for domain in DOMAINS:
    print(f'\nExporting {domain}...')
    !python export.py --domain {domain} --adapter_path adapters/{domain}_lora.pt
    print(f'  Done.')

In [ ]:
# Step 9: Convert to GGUF
# Clone llama.cpp for conversion
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama_cpp
!pip install -q -r /content/llama_cpp/requirements.txt

import os
os.makedirs('gguf', exist_ok=True)

for domain in DOMAINS:
    src = f'exports/cairn-{domain}'
    dst = f'gguf/cairn-{domain}-f16.gguf'
    dst_q4 = f'gguf/cairn-{domain}-q4_k_m.gguf'

    if not os.path.exists(f'{src}/model.safetensors'):
        print(f'Skipping {domain} - no model file')
        continue

    print(f'\nConverting {domain}...')
    !python /content/llama_cpp/convert_hf_to_gguf.py {src} --outfile {dst} --outtype f16

    if os.path.exists(dst):
        size = os.path.getsize(dst) / 1e6
        print(f'  {dst}: {size:.0f} MB')
    else:
        print(f'  WARNING: {dst} not created')

In [ ]:
# Step 10: Download
!zip -r /content/cairn-models.zip gguf/ adapters/

from google.colab import files
files.download('/content/cairn-models.zip')

print('\nDone! After download:')
print('  1. Unzip cairn-models.zip')
print('  2. Copy gguf/*.gguf to ml-pipeline/gguf/')
print('  3. For each domain: ollama create cairn-{domain} -f ollama/Modelfile.{domain}')